<a href="https://colab.research.google.com/github/euangd/openclient/blob/main/Qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/brian31415/prismlm-ternary-fork.git
%cd prismlm-ternary-fork
!pip install -r requirements.txt

In [ ]:
# Import library and run the model
# Note: Adjust the import paths or model weights file depending on your local setup
try:
    from model import PrismLM
    print("PrismLM imported successfully!")
except ImportError:
    print("Could not import PrismLM. Please check the repository structure or documentation.")

In [ ]:
# Install huggingface_hub and llama-cpp-python for serving the GGUF model
!pip install huggingface-hub llama-cpp-python

In [ ]:
from huggingface_hub import hf_hub_download

# Download the specific GGUF model from Hugging Face
model_repo = 'OS-Software/Ternary-Bonsai-2-27B-Uncensored-Heretic-GGUF'
# We will download a typical Q4_K_M quantization if available, or please adjust the filename as necessary
model_filename = 'Ternary-Bonsai-2-27B-Uncensored-Heretic.Q4_K_M.gguf'

try:
    model_path = hf_hub_download(repo_id=model_repo, filename=model_filename)
    print(f'Model downloaded successfully to: {model_path}')
except Exception as e:
    print(f'Error downloading model: {e}')
    print('Please verify the exact GGUF filename on Hugging Face.')

In [ ]:
# Run a local OpenAI-compatible API server using llama_cpp.server
# This will run in the background so OpenCode or other tools can connect to it
import subprocess
import time

# Start the API server on port 8000
server_process = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8000'
])

# Give the server a moment to initialize
time.sleep(5)
print('API server started on http://localhost:8000. You can now configure OpenCode to point to this endpoint.')

### Configuring OpenCode
To connect OpenCode (or any OpenAI-compatible client) to your local Ternary Bonsai API server, use the following endpoint details:

* **API Base URL**: `http://localhost:8000/v1`
* **Model Name**: Use the path or name of the model (e.g., `Ternary-Bonsai-2-27B-Uncensored-Heretic.Q4_K_M.gguf` or simply leave it default if OpenCode auto-detects).
* **API Key**: Since it's running locally without authentication, you can pass any dummy key (e.g., `not-needed` or `sk-1234`).

In [ ]:
# Verification: Query the local server using standard OpenAI client format
import requests

try:
    response = requests.get('http://localhost:8000/v1/models')
    if response.status_code == 200:
        print('Successfully connected to the API Server!')
        print('Available Models:', response.json())
    else:
        print(f'Server responded with status code: {response.status_code}')
except Exception as e:
    print(f'Could not connect to the API server: {e}')